# Scraping Laptop di Marketplace Indonesia

Notebook ini melakukan **loop** scraping untuk beberapa *jenis/brand laptop* (ASUS, Acer, Lenovo, HP, Dell, MSI, MacBook, dst) di beberapa marketplace (Tokopedia, Shopee) memakai adapter yang sudah ada di folder ini (`tokopedia_adapter.py`, `shopee_adapter.py`, dst), lalu menggabungkan semua hasil menjadi **satu file CSV dan satu file JSON**.

Alur:
1. Loop tiap kombinasi *(keyword laptop, situs)* -> scrape -> kumpulkan ke satu list.
2. Filter listing yang sebenarnya aksesoris (bukan unit laptop) via `category_filter.py`.
3. Tandai listing yang mencurigakan (indikasi scam) via `scam_filter.py`.
4. Export gabungan semua situs & semua keyword ke `laptop_listings_all.csv` dan `laptop_listings_all.json`.

> **Catatan penting soal Playwright + Jupyter di Windows:** kernel Jupyter memaksa event loop asyncio bertipe `Selector` (demi kompatibilitas zmq), sedangkan Playwright *sync API* butuh event loop bertipe `Proactor` untuk bisa spawn subprocess browser. Policy asyncio itu berlaku *process-wide*, jadi menjalankannya di thread lain di proses kernel yang sama tetap gagal. Solusinya: tiap panggilan scrape dijalankan di **proses OS terpisah** lewat script `scrape_worker.py`, dipanggil dari notebook via `subprocess`.

## 0. Setup awal (sekali saja)

Jalankan sel di bawah kalau dependency/browser Playwright belum terpasang.

In [1]:
# %pip install -r requirements.txt pandas
# %pip install playwright && playwright install chromium

## 1. Import module scraper yang sudah ada

In [ ]:
import sys, os, time, json, random, subprocess, tempfile

# pastikan folder notebook ini (berisi *_adapter.py dkk) ada di sys.path
PROJECT_DIR = os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from schema import LaptopListing
from category_filter import filter_accessories
from scam_filter import filter_listings
from exporter import export_csv, export_json
from spec_enricher import enrich_listings

print("modul scraper siap dipakai.")

## 2. Konfigurasi: situs & jenis laptop yang mau di-loop

`LAPTOP_KEYWORDS` berisi keyword pencarian per brand/jenis laptop yang umum dijual di Indonesia. Tambah/hapus sesuai kebutuhan.

In [47]:
SITES = ["tokopedia" ]

LAPTOP_KEYWORDS = [
    "laptop asus",]
    # "laptop acer",
    # "laptop lenovo",
    # "laptop hp",
    # "laptop dell",
    # "laptop msi",
    # "laptop axioo",
    # "macbook",]


MAX_PAGES = 1       # halaman hasil pencarian per keyword per situs
DROP_SCAMS = True      # True kalau mau langsung buang listing yang dicurigai scam
DELAY_BETWEEN_RUNS = (2.0, 4.0)  # jeda antar kombinasi keyword x situs (detik)
WORKER_TIMEOUT_S = 180  # batas waktu tiap proses worker (detik)


## 3. Helper: jalankan scrape (Playwright) di proses terpisah

`scrape_worker.py` di folder ini menerima `(site, keyword, max_pages, out_path)` lewat argumen CLI, menjalankan adapter yang sesuai, lalu menulis hasilnya sebagai JSON ke `out_path`. Notebook cukup membaca file itu kembali.

In [48]:
def scrape_isolated(site_name: str, keyword: str, max_pages: int):
    """Jalankan scrape_worker.py sebagai proses Python baru supaya Playwright
    punya event loop asyncio sendiri, terpisah dari event loop kernel Jupyter."""
    with tempfile.TemporaryDirectory() as td:
        out_path = os.path.join(td, "out.json")
        cmd = [sys.executable, "scrape_worker.py", site_name, keyword, str(max_pages), out_path]
        proc = subprocess.run(
            cmd, cwd=PROJECT_DIR, capture_output=True, text=True, timeout=WORKER_TIMEOUT_S
        )
        if proc.stdout:
            print(proc.stdout.strip())
        if proc.returncode != 0:
            print(proc.stderr[-2000:])
            raise RuntimeError(f"scrape_worker gagal (exit {proc.returncode}) untuk [{site_name}] '{keyword}'")
        with open(out_path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        return [LaptopListing(**d) for d in raw]

## 4. Loop scraping: tiap jenis laptop x tiap situs

In [70]:
all_listings = []

for keyword in LAPTOP_KEYWORDS:
    for site_name in SITES:
        print(f"[{site_name}] scraping '{keyword}' ({MAX_PAGES} halaman)...")
        try:
            listings = scrape_isolated(site_name, keyword, MAX_PAGES)
        except Exception as e:
            print(f"  !! gagal scrape [{site_name}] '{keyword}': {e}")
            listings = []
        print(f"  -> dapat {len(listings)} listing")
        all_listings.extend(listings)
        time.sleep(random.uniform(*DELAY_BETWEEN_RUNS))

print(f"\nTotal listing mentah terkumpul dari semua keyword & situs: {len(all_listings)}")

[tokopedia] scraping 'laptop asus' (1 halaman)...
[intercept_json] tangkap 'SearchProduct', items=?
[intercept_json][debug] url respons: https://gql.tokopedia.com/graphql/SearchProductV5Query
[intercept_json][debug] top-level keys respons: <class 'list'>
[tokopedia][debug] contoh raw_card keys: ['oldID', 'id', 'ttsProductID', 'name', 'url', 'applink', 'mediaURL', 'shop', 'stock', 'badge', 'price', 'freeShipping', 'labelGroups', 'labelGroupsVariant', 'category', 'rating', 'wishlist', 'ads', 'meta', '__typename']
[tokopedia][debug] contoh shop keys: ['oldID', 'id', 'ttsSellerID', 'name', 'url', 'city', 'tier', '__typename']
  -> dapat 60 listing

Total listing mentah terkumpul dari semua keyword & situs: 60


## 5. Bersihkan hasil: buang aksesoris, tandai dugaan scam

In [71]:
all_listings = filter_accessories(all_listings)
all_listings = filter_listings(all_listings, drop_scams=DROP_SCAMS)

n_scam = sum(1 for l in all_listings if l.is_suspected_scam)
print(f"total akhir={len(all_listings)} suspected_scam={n_scam}")

filtered out 25 non-laptop listings
total akhir=35 suspected_scam=0


In [65]:
all_listings

[LaptopListing(source='tokopedia', source_id='103175613088', url='https://www.tokopedia.com/royalltech/laptop-asus-vivobook-f1504vap-intel-core-i5-120u-i5-1235u-16gb-512gb-ssd-15-6-fhd-ips-windows-11-f1504za-f1502za-1730412882049598967-1734913183835784695?extParam=ivf%3Dfalse%26keyword%3Dlaptop+asus%26search_id%3D20260829101233352FE25DAF686B352DVM%26src%3Dsearch', title='Laptop ASUS Vivobook F1504VAP Intel Core i5-120U 16GB 512GB SSD 15.6”FHD IPS Windows 11 – F1504ZA / F1502ZA', brand='Asus', model='F1504VAP', cpu='Intel Core i5-120U', ram_gb=16, storage_gb=512, gpu=None, screen_size_in=15.6, price_idr=11309000, original_price_idr=None, condition='new', seller_name='royalltech', seller_rating=4.6, seller_num_reviews=None, seller_is_official=False, sold_count=500, location='Jakarta Utara', scraped_at='2026-08-29T10:13:00.491458', is_suspected_scam=False, scam_reasons=[]),
 LaptopListing(source='tokopedia', source_id='14997359922', url='https://www.tokopedia.com/hosanacomp/laptop-asus-vi

In [ ]:
all_listings = enrich_listings(all_listings)


## 6. Gabungkan semua hasil jadi satu file (CSV + JSON)

In [ ]:
OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PREFIX = "laptop_listings_Tokopedia_2023-06-20(6)"  # prefix nama file output (tanpa ekstensi)
csv_path = os.path.join(OUT_DIR, f"{OUT_PREFIX}.csv")
json_path = os.path.join(OUT_DIR, f"{OUT_PREFIX}.json")

export_csv(all_listings, csv_path)
export_json(all_listings, json_path)

print(f"tersimpan: {csv_path} ({len(all_listings)} baris)")
print(f"tersimpan: {json_path} ({len(all_listings)} baris)")

## 7. Preview hasil gabungan

In [69]:
import pandas as pd

df = pd.read_csv(csv_path)
print(df.shape)
df

(35, 23)


,source,source_id,url,title,brand,model,cpu,ram_gb,storage_gb,gpu,...,condition,seller_name,seller_rating,seller_num_reviews,seller_is_official,sold_count,location,scraped_at,is_suspected_scam,scam_reasons
0,tokopedia,103175613088,https://www.tokopedia.com/royalltech/laptop-as...,Laptop ASUS Vivobook F1504VAP Intel Core i5-12...,Asus,F1504VAP,Intel Core i5-120U,16.0,512.0,NaN,...,new,royalltech,4.6,NaN,False,500,Jakarta Utara,2026-08-29T10:13:00.491458,False,NaN
1,tokopedia,14997359922,https://www.tokopedia.com/hosanacomp/laptop-as...,LAPTOP ASUS VIVOBOOK GO E410KA - N4500 8GB 256...,Asus,E410KA,Intel N4500,8.0,1024.0,NaN,...,new,HOSANA COMPUTER,5.0,NaN,False,20,Jakarta Pusat,2026-08-29T10:13:00.491843,False,NaN
2,tokopedia,1395582448,https://www.tokopedia.com/hosanacomp/laptop-as...,LAPTOP ASUS A409JP i5 - 1035G1 4GB 512GB SSD M...,Asus,A409JP,NaN,4.0,512.0,NaN,...,new,HOSANA COMPUTER,5.0,NaN,False,1,Jakarta Pusat,2026-08-29T10:13:00.492044,False,NaN
3,tokopedia,102316114277,https://www.tokopedia.com/gateway/asus-zenbook...,ASUS ZENBOOK 14 OLED UM3406HA TOUCH RYZEN 7 88...,Asus,UM3406HA,AMD Ryzen 7 8840,16.0,1024.0,NaN,...,new,Gateway Indonesia Comp,5.0,NaN,False,50,Jakarta Utara,2026-08-29T10:13:00.492291,False,NaN
4,tokopedia,2041423306,https://www.tokopedia.com/hosanacomp/laptop-as...,LAPTOP ASUS A416JAO - i3 1005G1 8GB 1TB + 256G...,Asus,A416JAO,NaN,8.0,256.0,NaN,...,new,HOSANA COMPUTER,5.0,NaN,False,2,Jakarta Pusat,2026-08-29T10:13:00.496262,False,NaN
5,tokopedia,2542238886,https://www.tokopedia.com/hosanacomp/laptop-as...,LAPTOP ASUS A509FA - N4305U 8GB 256GB SSD 15.6...,Asus,A509FA,NaN,8.0,256.0,NaN,...,new,HOSANA COMPUTER,5.0,NaN,False,10,Jakarta Pusat,2026-08-29T10:13:00.497400,False,NaN
6,tokopedia,103106494804,https://www.tokopedia.com/gateway/asus-tuf-gam...,ASUS TUF GAMING A15 FA506NCG RYZEN 7 7445HS RT...,Asus,FA506NCG,AMD Ryzen 7 7445HS,32.0,1024.0,RTX3050,...,new,Gateway Indonesia Comp,5.0,NaN,False,100,Jakarta Utara,2026-08-29T10:13:00.497833,False,NaN
7,tokopedia,103629770816,https://www.tokopedia.com/gateway/hp-pc-aio-24...,ASUS TUF GAMING A16 FA607NUQ RYZEN 7 170 RTX40...,Asus,FA607NUQ,AMD Ryzen 7 170,32.0,1024.0,RTX4050,...,new,Gateway Indonesia Comp,5.0,NaN,False,5,Jakarta Utara,2026-08-29T10:13:00.498170,False,NaN
8,tokopedia,103568217966,https://www.tokopedia.com/gateway/asus-zenbook...,ASUS VIVOBOOK 15 F1504ZA TOUCH CORE I7 1255 16...,Asus,F1504ZA,Intel Core i7-1255,16.0,512.0,NaN,...,new,Gateway Indonesia Comp,5.0,NaN,False,50,Jakarta Utara,2026-08-29T10:13:00.498514,False,NaN
9,tokopedia,15851909520,https://www.tokopedia.com/technocentral/asus-v...,[BEST SELLER] LAPTOP ASUS VIVOBOOK GO 14 E1404...,Asus,E1404FA,AMD Ryzen 5 7520,16.0,512.0,NaN,...,new,Techno Central,5.0,NaN,False,26,Bogor,2026-08-29T10:13:00.498768,False,NaN


In [86]:
# ringkasan cepat: jumlah listing per brand & per situs
display(df.groupby(["source", "brand"]).size().unstack(fill_value=0))
df.isnull().sum().sort_values(ascending=False)

brand,Acer,Apple,Asus,Axioo,Dell,Hp,Lenovo,Msi
source,,,,,,,,
tokopedia,209,2,199,237,36,23,194,70


scam_reasons          970
original_price_idr    970
seller_num_reviews    970
screen_size_in        864
gpu                   745
model                 558
storage_gb            450
cpu                   186
ram_gb                106
seller_rating          35
sold_count             29
location                1
is_suspected_scam       0
scraped_at              0
seller_is_official      0
source                  0
seller_name             0
condition               0
source_id               0
brand                   0
title                   0
url                     0
price_idr               0
dtype: int64